# 예제 02. Dropout 적용
빅데이터프로그래밍 · 9주차

## 목표
- Dropout이 뉴런을 임시로 끄는 것을 직접 본다
- `model.train()` 과 `model.eval()` 의 차이를 확인한다
- 같은 조건에서 과적합이 줄어드는 것을 비교한다

학습 중 일부 뉴런을 임시로 사용하지 않아, 특정 뉴런에 지나치게 의존하는 것을 줄입니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. Dropout이 하는 일을 숫자로 보기
확률 0.5로 절반 정도를 0으로 만들고, 남은 값은 2배로 키웁니다 (전체 크기를 유지하려고).


In [ ]:
drop = nn.Dropout(p=0.5)
x = torch.ones(2, 10)

drop.train()                       # 학습 모드
print("학습 모드 (일부가 0):")
print(drop(x))

drop.eval()                        # 평가 모드
print("\n평가 모드 (전부 그대로):")
print(drop(x))


`eval()` 에서는 Dropout이 **아무것도 하지 않습니다.** 예측할 때 뉴런을 끄면 안 되기 때문입니다.


In [ ]:
# 매번 다른 뉴런이 꺼집니다
drop.train()
for i in range(3):
    print(f"{i+1}회:", drop(torch.ones(1, 10)).squeeze().tolist())


## 2. train() 과 eval() 을 잊으면
학습 루프에서 이 두 줄을 빼면 결과가 조용히 나빠집니다. 오류가 나지 않아 찾기 어렵습니다.


In [ ]:
m = nn.Sequential(nn.Linear(10, 10), nn.Dropout(0.5))
x = torch.ones(1, 10)

m.train()
outs = [m(x).sum().item() for _ in range(5)]
print("train 모드에서 5번:", [round(v, 3) for v in outs], "← 매번 다릅니다")

m.eval()
outs = [m(x).sum().item() for _ in range(5)]
print("eval  모드에서 5번:", [round(v, 3) for v in outs], "← 항상 같습니다")


## 3. 데이터와 공통 함수


In [ ]:
transform = transforms.ToTensor()
full_train = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_set   = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(Subset(full_train, range(2000)), batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=256, shuffle=False)

loss_fn = nn.CrossEntropyLoss()

def measure(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def train(model, epochs=30, lr=1e-3, quiet=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()                       # 매 epoch 시작에 train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append((*measure(model, train_loader), *measure(model, test_loader)))
        if not quiet and (epoch % 10 == 0 or epoch == 1):
            print(f"  epoch {epoch:2d}  학습 {hist[-1][1]:.4f}  검증 {hist[-1][3]:.4f}")
    return model, hist


## 4. Dropout 없는 모델과 있는 모델
Dropout은 보통 **Flatten 뒤 Linear 사이**에 넣습니다. Conv 층 사이에는 낮은 확률로 씁니다.


In [ ]:
class PlainCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.ReLU(),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class DropoutCNN(nn.Module):
    def __init__(self, n_classes=10, p=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p),                          # Flatten 직후
            nn.Linear(64*7*7, 256), nn.ReLU(),
            nn.Dropout(p),                          # 은닉층 뒤
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


print("기본 CNN 학습");    torch.manual_seed(42); plain, plain_hist = train(PlainCNN(), quiet=False)
print("\nDropout CNN 학습"); torch.manual_seed(42); dropm, drop_hist = train(DropoutCNN(), quiet=False)


## 5. 비교


In [ ]:
import pandas as pd

def summary(name, hist):
    return {"모델": name,
            "학습 정확도": round(hist[-1][1], 4),
            "검증 정확도": round(hist[-1][3], 4),
            "차이": round(hist[-1][1] - hist[-1][3], 4),
            "최고 검증": round(max(h[3] for h in hist), 4)}

pd.DataFrame([summary("기본 CNN", plain_hist), summary("Dropout CNN", drop_hist)])


In [ ]:
xs = range(1, len(plain_hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(xs, [h[2] for h in plain_hist], label="기본 · 검증 손실")
ax[0].plot(xs, [h[2] for h in drop_hist],  label="Dropout · 검증 손실")
ax[0].set_title("검증 손실"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h[1]-h[3] for h in plain_hist], label="기본")
ax[1].plot(xs, [h[1]-h[3] for h in drop_hist],  label="Dropout")
ax[1].set_title("학습 − 검증 정확도 차이"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. 확률을 바꿔 보기
너무 크면 학습 자체가 어려워집니다. 0.2 ~ 0.5 사이에서 고릅니다.


In [ ]:
rows = []
for p in [0.0, 0.2, 0.5, 0.8]:
    torch.manual_seed(42)
    _, h = train(DropoutCNN(p=p), epochs=20)
    rows.append({"p": p, "학습 정확도": round(h[-1][1], 4),
                 "검증 정확도": round(h[-1][3], 4),
                 "차이": round(h[-1][1]-h[-1][3], 4)})
pd.DataFrame(rows)


## 직접 해보기
1. Conv 층 사이에 `nn.Dropout2d(0.25)` 를 넣으면 어떻게 되나요?
2. `model.train()` 을 빼고 학습하면 결과가 어떻게 달라지나요?


In [ ]:
# 여기에 작성하세요
